<a href="https://colab.research.google.com/github/Cshoga/Intro-to-ML_Summative/blob/main/Student_Dropout_ML_DL_Comparison_Celine_Shoga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning and Deep Learning for Predicting Student Dropout and Academic Success

This notebook follows the same experiment-based style as the previous Malaria Diagnosis CNN notebook, but it is adapted for an **education-focused tabular dataset**.

The project compares traditional Machine Learning models built with **Scikit-learn** against Deep Learning models built with **TensorFlow/Keras** using the **Sequential API**, **Functional API**, and **tf.data API**.

**Project problem:** Predict whether a student will **Dropout**, remain **Enrolled**, or **Graduate** using academic, demographic, and socioeconomic indicators.

**Dataset:** Students Dropout and Academic Success dataset from Kaggle/UCI.

## Configuration

This section prepares the notebook environment. In Colab, use the Kaggle API key `kaggle.json` to download the dataset directly and make the notebook reproducible.

In [ ]:
# ============================================================
# 1. INSTALL AND CONFIGURE KAGGLE
# ============================================================

!pip install -q kaggle

import os
import warnings
warnings.filterwarnings("ignore")

# Upload kaggle.json only if it is not already available
if not os.path.exists("/root/.kaggle/kaggle.json"):
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()

    os.makedirs("/root/.kaggle", exist_ok=True)
    !cp kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

print("Kaggle API is ready.")

Please upload your kaggle.json file:


In [ ]:
# ============================================================
# 2. DOWNLOAD DATASET FROM KAGGLE
# ============================================================

# Dataset: Higher Education Predictors of Student Retention
!kaggle datasets download -d thedevastator/higher-education-predictors-of-student-retention -p /content/student_dropout_data

# Unzip dataset
!unzip -o /content/student_dropout_data/higher-education-predictors-of-student-retention.zip -d /content/student_dropout_data

# Show downloaded files
print("Files in dataset folder:")
print(os.listdir("/content/student_dropout_data"))

## Populating Namespaces

This section imports all libraries required for data analysis, visualization, machine learning, deep learning, and evaluation.

In [ ]:
# ============================================================
# 3. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing and splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.pipeline import Pipeline

# Machine Learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import Sequential, Model, Input
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## Prepare Dataset

This section loads the dataset, checks its structure, and performs basic data understanding.

In [ ]:
# ============================================================
# 4. LOAD DATASET
# ============================================================

data_dir = "/content/student_dropout_data"

csv_files = [file for file in os.listdir(data_dir) if file.endswith(".csv")]
print("CSV files found:", csv_files)

# Load the first CSV file automatically
csv_path = os.path.join(data_dir, csv_files[0])
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# ============================================================
# 5. DATASET INFORMATION
# ============================================================

print("Dataset Information:")
df.info()

print("\nMissing values per column:")
display(df.isnull().sum())

print("\nTarget distribution:")
display(df["Target"].value_counts())

### Insight

The target variable has three classes: **Dropout**, **Enrolled**, and **Graduate**. This makes the task a multiclass classification problem. Before modeling, it is important to check whether the classes are balanced because class imbalance can affect precision, recall, and F1-score.

In [ ]:
# ============================================================
# 6. TARGET DISTRIBUTION VISUALIZATION
# ============================================================

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Target")
plt.title("Distribution of Student Outcomes")
plt.xlabel("Student Outcome")
plt.ylabel("Number of Students")
plt.show()

class_percentages = df["Target"].value_counts(normalize=True) * 100
print("Class percentages:")
display(class_percentages)

In [ ]:
# ============================================================
# 7. BASIC STATISTICAL SUMMARY
# ============================================================

display(df.describe().T)

## Data Preprocessing and Feature Engineering

The dataset contains numeric and categorical-looking columns. For this project, the target variable is encoded into numbers:

- Dropout = 0
- Enrolled = 1
- Graduate = 2

The features are separated from the target, and the data is split into training, validation, and test sets.

In [ ]:
# ============================================================
# 8. PREPROCESSING
# ============================================================

# Make a copy to protect the original dataframe
data = df.copy()

# Remove leading/trailing spaces from column names
data.columns = data.columns.str.strip()

# Encode target labels
target_encoder = LabelEncoder()
data["Target_encoded"] = target_encoder.fit_transform(data["Target"])

print("Target encoding:")
for label, encoded in zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)):
    print(f"{label}: {encoded}")

# Separate features and target
X = data.drop(columns=["Target", "Target_encoded"])
y = data["Target_encoded"]

# Convert any object columns into numeric dummy variables if present
X = pd.get_dummies(X, drop_first=True)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# ============================================================
# 9. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First split: train+validation and test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Second split: train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_val
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

In [ ]:
# ============================================================
# 10. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

num_classes = len(np.unique(y))
n_features = X_train_scaled.shape[1]

print("Number of features:", n_features)
print("Number of classes:", num_classes)

## Helper Functions for Training, Curves, and Evaluation

This section follows the experiment-based structure from the previous notebook. Helper functions reduce repeated code and make the experiments easier to compare.

In [ ]:
# ============================================================
# 11. HELPER FUNCTIONS
# ============================================================

results = []

def evaluate_model(model_name, y_true, y_pred):
    """Calculate and store model evaluation metrics."""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

    print(f"\n{model_name} Results")
    print("-" * 50)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=target_encoder.classes_, zero_division=0))

def plot_confusion_matrix(y_true, y_pred, title):
    """Plot a confusion matrix for multiclass classification."""
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_encoder.classes_)
    disp.plot(cmap="Blues", xticks_rotation=45)
    plt.title(title)
    plt.show()

def plot_learning_curves(history, title):
    """Plot accuracy and loss curves for TensorFlow models."""
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(title + " - Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(title + " - Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

## Baseline Traditional Machine Learning Models

The first group of experiments uses Scikit-learn models. These models provide a strong baseline for tabular education data.

## Experiment 1: Logistic Regression

In [ ]:
# ============================================================
# EXPERIMENT 1: LOGISTIC REGRESSION
# ============================================================

log_reg = LogisticRegression(max_iter=1000, random_state=SEED)

log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

evaluate_model("Logistic Regression", y_test, y_pred_lr)
plot_confusion_matrix(y_test, y_pred_lr, "Confusion Matrix - Logistic Regression")

### Experiment 1 Insight

Logistic Regression is used as a baseline because it is simple, interpretable, and commonly used for classification. If this model performs well, it means the relationship between the features and the student outcome may be partly captured by linear decision boundaries.

## Experiment 2: Random Forest

In [ ]:
# ============================================================
# EXPERIMENT 2: RANDOM FOREST
# ============================================================

random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=SEED,
    class_weight="balanced"
)

random_forest.fit(X_train, y_train)
y_pred_rf = random_forest.predict(X_test)

evaluate_model("Random Forest", y_test, y_pred_rf)
plot_confusion_matrix(y_test, y_pred_rf, "Confusion Matrix - Random Forest")

### Experiment 2 Insight

Random Forest is useful for this dataset because it can capture non-linear relationships between academic, demographic, and socioeconomic variables. It also handles feature interactions better than Logistic Regression.

## Experiment 3: Support Vector Machine

In [ ]:
# ============================================================
# EXPERIMENT 3: SUPPORT VECTOR MACHINE
# ============================================================

svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=SEED,
    class_weight="balanced"
)

svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

evaluate_model("Support Vector Machine", y_test, y_pred_svm)
plot_confusion_matrix(y_test, y_pred_svm, "Confusion Matrix - Support Vector Machine")

### Experiment 3 Insight

The RBF SVM tests whether a more flexible non-linear boundary improves prediction. Scaling is important for SVM because it is sensitive to feature magnitude.

## Experiment 4: Gradient Boosting

In [ ]:
# ============================================================
# EXPERIMENT 4: GRADIENT BOOSTING
# ============================================================

gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=SEED
)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

evaluate_model("Gradient Boosting", y_test, y_pred_gb)
plot_confusion_matrix(y_test, y_pred_gb, "Confusion Matrix - Gradient Boosting")

### Experiment 4 Insight

Gradient Boosting builds models sequentially and corrects previous errors. It often performs strongly on structured/tabular datasets, so it is an important comparison against the deep learning models.

## Preparing Data for Deep Learning with tf.data

The assignment requires the use of **TensorFlow tf.data API**. This section converts NumPy arrays into efficient TensorFlow datasets.

In [ ]:
# ============================================================
# 12. CREATE TF.DATA PIPELINES
# ============================================================

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((X_train_scaled.astype("float32"), y_train.values.astype("int32")))
val_ds = tf.data.Dataset.from_tensor_slices((X_val_scaled.astype("float32"), y_val.values.astype("int32")))
test_ds = tf.data.Dataset.from_tensor_slices((X_test_scaled.astype("float32"), y_test.values.astype("int32")))

train_ds = train_ds.shuffle(buffer_size=len(X_train_scaled), seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("tf.data pipelines are ready.")

In [ ]:
# ============================================================
# 13. DEEP LEARNING HELPER FUNCTIONS
# ============================================================

def compile_and_train(model, model_name, learning_rate=0.001, epochs=50):
    """Compile and train a TensorFlow model."""
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    print(model.summary())

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[early_stop],
        verbose=1
    )

    y_proba = model.predict(test_ds)
    y_pred = np.argmax(y_proba, axis=1)

    evaluate_model(model_name, y_test, y_pred)
    plot_learning_curves(history, model_name)
    plot_confusion_matrix(y_test, y_pred, f"Confusion Matrix - {model_name}")

    return model, history, y_proba, y_pred

## Deep Learning Experiments

The next experiments use TensorFlow/Keras. The first models use the **Sequential API**, while later models use the **Functional API**.

## Experiment 5: Sequential Neural Network

In [ ]:
# ============================================================
# EXPERIMENT 5: SEQUENTIAL NEURAL NETWORK
# ============================================================

seq_model_1 = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    Dropout(0.30),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

seq_model_1, history_seq_1, y_proba_seq_1, y_pred_seq_1 = compile_and_train(
    seq_model_1,
    "Sequential NN",
    learning_rate=0.001,
    epochs=50
)

### Experiment 5 Insight

This Sequential neural network is the baseline deep learning model. It tests whether a simple dense network can learn patterns from the scaled student features.

## Experiment 6: Deeper Sequential Neural Network

In [ ]:
# ============================================================
# EXPERIMENT 6: DEEPER SEQUENTIAL NEURAL NETWORK
# ============================================================

seq_model_2 = Sequential([
    Dense(128, activation="relu", input_shape=(n_features,)),
    Dropout(0.30),
    Dense(64, activation="relu"),
    Dropout(0.20),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

seq_model_2, history_seq_2, y_proba_seq_2, y_pred_seq_2 = compile_and_train(
    seq_model_2,
    "Deeper Sequential NN",
    learning_rate=0.001,
    epochs=50
)

### Experiment 6 Insight

This experiment checks whether adding more hidden layers improves performance. If validation accuracy does not improve or validation loss increases, the deeper model may be overfitting.

## Experiment 7: Functional API Neural Network

In [ ]:
# ============================================================
# EXPERIMENT 7: FUNCTIONAL API NEURAL NETWORK
# ============================================================

inputs = Input(shape=(n_features,))
x = Dense(128)(inputs)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = Dropout(0.30)(x)

x = Dense(64)(x)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = Dropout(0.20)(x)

outputs = Dense(num_classes, activation="softmax")(x)

functional_model = Model(inputs=inputs, outputs=outputs)

functional_model, history_func, y_proba_func, y_pred_func = compile_and_train(
    functional_model,
    "Functional API NN",
    learning_rate=0.001,
    epochs=50
)

### Experiment 7 Insight

The Functional API allows more flexible model design. Batch normalization is included to stabilize learning and improve convergence.

## Experiment 8: Functional API with Lower Learning Rate

In [ ]:
# ============================================================
# EXPERIMENT 8: FUNCTIONAL API WITH LOWER LEARNING RATE
# ============================================================

inputs_lr = Input(shape=(n_features,))
x = Dense(128)(inputs_lr)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = Dropout(0.30)(x)

x = Dense(64, activation="relu")(x)
x = Dropout(0.20)(x)

outputs_lr = Dense(num_classes, activation="softmax")(x)

functional_lr_model = Model(inputs=inputs_lr, outputs=outputs_lr)

functional_lr_model, history_func_lr, y_proba_func_lr, y_pred_func_lr = compile_and_train(
    functional_lr_model,
    "Functional API NN LR=0.0005",
    learning_rate=0.0005,
    epochs=60
)

### Experiment 8 Insight

This experiment tests whether a lower learning rate improves convergence. A lower learning rate may train more slowly but can sometimes produce smoother validation curves.

## Experiment Comparison Table

In [ ]:
# ============================================================
# 14. EXPERIMENT COMPARISON TABLE
# ============================================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)

display(results_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="F1-score", y="Model")
plt.title("Model Comparison by Weighted F1-score")
plt.xlabel("Weighted F1-score")
plt.ylabel("Model")
plt.show()

## ROC Curve for Best Models

ROC curves are created using predicted probabilities. For multiclass classification, each class is treated using a one-vs-rest approach.

In [ ]:
# ============================================================
# 15. MULTICLASS ROC CURVES
# ============================================================

def plot_multiclass_roc(y_true, y_score, model_name):
    y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))

    plt.figure(figsize=(8, 6))

    for i, class_name in enumerate(target_encoder.classes_):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{class_name} AUC = {roc_auc:.3f}")

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"Multiclass ROC Curve - {model_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

# ROC for selected traditional ML models
plot_multiclass_roc(y_test, log_reg.predict_proba(X_test_scaled), "Logistic Regression")
plot_multiclass_roc(y_test, random_forest.predict_proba(X_test), "Random Forest")
plot_multiclass_roc(y_test, gb_model.predict_proba(X_test), "Gradient Boosting")

# ROC for best deep learning model candidates
plot_multiclass_roc(y_test, y_proba_func, "Functional API NN")
plot_multiclass_roc(y_test, y_proba_func_lr, "Functional API NN LR=0.0005")

## Feature Importance for Tree-Based Model

Feature importance helps explain which variables contributed most to the Random Forest predictions. This supports the discussion and critical analysis sections of the report.

In [ ]:
# ============================================================
# 16. FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest.feature_importances_
}).sort_values(by="Importance", ascending=False)

display(feature_importance.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(15), x="Importance", y="Feature")
plt.title("Top 15 Important Features - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## Best Model and Error Analysis

This section identifies the best-performing model and supports critical error analysis by examining its confusion matrix.

In [ ]:
# ============================================================
# 17. BEST MODEL SUMMARY
# ============================================================

best_model_row = results_df.iloc[0]
print("Best model based on weighted F1-score:")
display(best_model_row)

print("\nFull comparison:")
display(results_df)

### Error Analysis Guidance

Use the confusion matrices to answer these questions in your report:

1. Which class was predicted most accurately?
2. Which class was confused most often?
3. Did the model confuse **Dropout** students with **Enrolled** students?
4. What are the risks of false negatives in a real university setting?
5. Which model gives the best balance between accuracy and recall?

In an education setting, recall for the **Dropout** class is especially important because missing at-risk students could prevent timely intervention.

## Overfitting, Underfitting, and Epoch Analysis

Learning curves help determine whether the neural networks are overfitting or underfitting.

- If training accuracy is high but validation accuracy is much lower, the model may be overfitting.
- If both training and validation accuracy are low, the model may be underfitting.
- If validation loss stops improving, early stopping prevents unnecessary training.

In [ ]:
# ============================================================
# 18. FINAL NOTES FOR REPORT
# ============================================================

print("Use the generated metrics, confusion matrices, ROC curves, learning curves, and feature importance plots in the written report.")
print("Recommended report discussion: compare whether traditional ML or DL performed better on tabular education data.")
print("It is common for Random Forest or Gradient Boosting to perform very strongly on tabular datasets.")

## References for Report

Use these references in IEEE style in the written report:

[1] M. V. Martins, D. Tolledo, J. Machado, L. M. T. Baptista, and V. Realinho, “Early prediction of student’s performance in higher education: A case study,” *Trends and Applications in Information Systems and Technologies*, 2021.

[2] UCI Machine Learning Repository, “Predict Students' Dropout and Academic Success,” 2021.

[3] Kaggle, “Higher Education Predictors of Student Retention,” Kaggle Dataset.

[4] F. Pedregosa et al., “Scikit-learn: Machine Learning in Python,” *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.

[5] M. Abadi et al., “TensorFlow: Large-scale machine learning on heterogeneous systems,” 2015.